In [17]:
# ==========================================
# 10. FILE HANDLING & CSV/JSON (Selected: Q122)
# ==========================================
# Question 122:
# Write a class CSVProcessor that can: (a) stream-process a large CSV row by row without loading it 
# fully, (b) filter rows by a user-supplied predicate, (c) transform values using a per-column mapping dict, and 
# (d) write the result to a new CSV. Avoid the csv module.
#
# Sample Input:  input_file = "raw_data.csv", output_file = "processed_data.csv"
#                Predicate: filter out rows where age < 25
#                Transformations: {"name": str.upper, "age": str}
# Sample Output: True (written to "processed_data.csv")

class CSVProcessor:
    def __init__(self, input_filepath, output_filepath):
        self.input_filepath = input_filepath
        self.output_filepath = output_filepath

    def process(self, predicate=None, transformers=None):
        with open(self.input_filepath, "r", encoding="utf-8") as f,open(self.output_filepath,'w') as r:
            head = f.readline().strip().split(',')
            r.write(",".join(head) + "\n")
            for i in f:
                val = i.strip().split(',')
                row = dict(zip(head,val))
                if predicate is not None and not predicate(row):
                    continue
                if transformers:
                    for k,v in transformers.items():
                        if k in row:
                            row[k] = v(row[k])
                # print(row)
                r.write(",".join(row.values()) + "\n")
                    
            return True


if __name__ == '__main__':
    # Test Question 122
    # Setup dummy input CSV
    in_path = r"C:\Work\Python\Assignment 2.1\Dataset\File Handling\Q122\test_raw_data.csv"
    out_path = r"C:\Work\Python\Assignment 2.1\Dataset\File Handling\Q122\test_processed_data.csv"
    with open(in_path, "w", encoding="utf-8") as f:
        f.write("name,age,city\nalice,30,New York\nbob,20,London\ncharlie,28,Paris\n")

    processor = CSVProcessor(in_path, out_path)
    res = processor.process(
        predicate=lambda row: int(row.get("age", 0)) >= 25,
        transformers={"name": str.upper}
    )
    print("Q122 Output:", res)

    # # Cleanup temporary test files
    # if os.path.exists(in_path): os.remove(in_path)
    # if os.path.exists(out_path): os.remove(out_path)

Q122 Output: True


In [27]:
import os
import json
import tempfile
from pathlib import Path
# ==========================================
# 10. FILE HANDLING & CSV/JSON (Selected: Q123)
# ==========================================
# Question 123:
# Write a function atomic_write(filepath, content) that writes content to a temp file first and then 
# atomically renames it to filepath. This prevents data corruption if the write is interrupted. Use the os and 
# tempfile modules.
#
# Sample Input:  filepath = "config.json", content = '{"status": "active"}'
# Sample Output: True

def atomic_write(filepath, content):

    with tempfile.NamedTemporaryFile(mode='w',delete=False) as f:
        f.write(content)
        temp_path = f.name
    os.replace(temp_path,filepath)
    return True

if __name__ == '__main__':
    # Test Question 123
    target_file = r"C:\Work\Python\Assignment 2.1\Dataset\File Handling\Q123\test_atomic_config.json"
    success = atomic_write(target_file, '{"status": "active"}')
    print("Q123 Output:", success)

    # Cleanup test file
    # if os.path.exists(target_file): os.remove(target_file)

Q123 Output: True


In [37]:
# ==========================================
# 10. FILE HANDLING & CSV/JSON (Selected: Q124)
# ==========================================
# Question 124:
# Write a program that recursively scans a directory tree, collects all .txt and .log files, and produces a 
# JSON report: {filename, path, size_bytes, line_count, word_count, top_5_words}. Use only os.walk and json 
# — no third-party libraries.
#
# Sample Input:  directory = "./sample_logs"
# Sample Output: [{"filename": "app.log", "path": "./sample_logs/app.log", "size_bytes": 1024, "line_count": 25, "word_count": 150, "top_5_words": [["error", 10], ["system", 5], ["failed", 4], ["user", 3], ["info", 2]]}]

import os
import json

def scan_and_report_files(directory_path):
    report = []

    for root, dirs, files in os.walk(directory_path):

        for file in files:

            if file.endswith(".txt") or file.endswith(".log"):

                path = os.path.join(root, file)
                with open(path, "r", encoding="utf-8") as f:
                    data = f.read()
                words = data.lower().split()
                counts = {
                    word: words.count(word)
                    for word in set(words)
                }
                top_5 = sorted(
                    counts.items(),
                    key=lambda x: x[1],
                    reverse=True
                )[:5]

                report.append({
                    "filename": file,
                    "path": path,
                    "size_bytes": os.path.getsize(path),
                    "line_count": len(data.splitlines()),
                    "word_count": len(words),
                    "top_5_words": top_5
                })

    return json.dumps(report, indent=4)


if __name__ == '__main__':
    test_dir = r"C:\Work\Python\Assignment 2.1\Dataset"

    sample_file_path = r"C:\Work\Python\Assignment 2.1\Dataset\File Handling\Q124\sample_app.log"

    with open(sample_file_path, "w", encoding="utf-8") as f:
        f.write(
            "error system error failed error user info "
            "system error failed\n"
        )

    report = scan_and_report_files(test_dir)
    with open(sample_file_path, "w", encoding="utf-8") as f:
            f.write(report)
    print("Q124 Output:", report)

    # Cleanup test setup
    # if os.path.exists(sample_file_path): os.remove(sample_file_path)
    # if os.path.exists(test_dir): os.rmdir(test_dir)

Q124 Output: [
    {
        "filename": "sample_app.log",
        "path": "C:\\Work\\Python\\Assignment 2.1\\Dataset\\File Handling\\Q124\\sample_app.log",
        "size_bytes": 63,
        "line_count": 1,
        "word_count": 10,
        "top_5_words": [
            [
                "error",
                4
            ],
            [
                "failed",
                2
            ],
            [
                "system",
                2
            ],
            [
                "info",
                1
            ],
            [
                "user",
                1
            ]
        ]
    }
]


In [ ]:
# ==========================================
# 10. FILE HANDLING & CSV/JSON (Selected: Q125)
# ==========================================
# Question 125:
# Write a function that merges multiple JSON files (each containing a list of records) into one, 
# deduplicating on a given key field. Log every duplicate found (with its source file) to a separate 
# deduplications.log file. Handle malformed JSON files gracefully.
#
# Sample Input:  file_list = ["file1.json", "file2.json"], key_field = "id"
# Sample Output: [{"id": 101, "name": "Alice"}, {"id": 102, "name": "Bob"}]

import json
import os

def merge_json_files(
    file_list,
    key_field,
    output_file=r"C:\Work\Python\Assignment 2.1\Dataset\File Handling\Q125\merged.json",
    log_file=r"C:\Work\Python\Assignment 2.1\Dataset\File Handling\Q125\deduplications.log"
):

    data = {}
    duplicates = []

    for file in file_list:

        try:
            with open(file, "r", encoding="utf-8") as f:
                records = json.load(f)

            for record in records:
                key = record[key_field]

                if key in data:
                    duplicates.append(
                        f"Duplicate {key} found in {file}\n"
                    )
                else:
                    data[key] = record

        except (json.JSONDecodeError, OSError, KeyError) as e:
            print(f"Skipping {file}: {e}")

    result = list(data.values())
    # print(data,result)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=4)

    with open(log_file, "w", encoding="utf-8") as f:
        f.writelines(duplicates)

    return result


if __name__ == '__main__':
    # Test Question 125
    f1, f2 = r"C:\Work\Python\Assignment 2.1\Dataset\File Handling\Q125\test_f1.json", r"C:\Work\Python\Assignment 2.1\Dataset\File Handling\Q125\test_f2.json"
    with open(f1, "w", encoding="utf-8") as f:
        json.dump([{"id": 101, "name": "Alice"}, {"id": 102, "name": "Bob"}], f)
    with open(f2, "w", encoding="utf-8") as f:
        json.dump([{"id": 101, "name": "Alice Duplicate"}, {"id": 103, "name": "Charlie"}], f)

    merged = merge_json_files([f1, f2], key_field="id")
    print("Q125 Output:", merged)

    # Cleanup temporary test files
    # for path in [f1, f2, "merged.json", "deduplications.log"]:
    #     if os.path.exists(path): os.remove(path)

{101: {'id': 101, 'name': 'Alice'}, 102: {'id': 102, 'name': 'Bob'}, 103: {'id': 103, 'name': 'Charlie'}} [{'id': 101, 'name': 'Alice'}, {'id': 102, 'name': 'Bob'}, {'id': 103, 'name': 'Charlie'}]
Q125 Output: [{'id': 101, 'name': 'Alice'}, {'id': 102, 'name': 'Bob'}, {'id': 103, 'name': 'Charlie'}]
